## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Parameter & Datasets

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/E2074/local-datasets/customer/appography/appography_data_dump/'
print(local_datasets)

/Users/E2074/analytics/customer/Appography/appography_data_dump
/Users/E2074/local-datasets/customer/appography/appography_data_dump/


In [4]:
current_date = datetime.now()

start_date = '20240630' ##(current_date - timedelta(days=2*30)).strftime('%Y%m%d')
end_date = current_date.strftime('%Y%m%d')
print(start_date, ' to ' ,end_date)

20240630  to  20240716


In [16]:
##users_app_list
all_data = pd.DataFrame()
looping_start_time = time.time()
date_range = pd.date_range(start=start_date, end=end_date)
for single_date in date_range:
    start_time = time.time()
    date_str = single_date.strftime('%Y%m%d')
    
    users_app_list = f"""
        WITH customer_app AS (
        
            SELECT
                yyyymmdd AS last_sync_date,
                id user_id, 
                -- appography_data as app_lists,
                CAST(json_parse(appography_data) AS ARRAY(VARCHAR)) AS app_lists,
                DENSE_RANK() over(partition by id order by yyyymmdd DESC) updated_seq
            FROM 
                customer.info_customer_installed_apps_immutable
            WHERE 
                yyyymmdd = '{date_str}'
            )
        
            SELECT 
                last_sync_date,
                user_id,
                LOWER(app_list) app_list
            FROM 
                (
                SELECT
                    last_sync_date,
                    user_id,
                    app_lists
                FROM 
                    customer_app
                WHERE 
                    updated_seq = 1
                ) t
            CROSS JOIN UNNEST(cast(app_lists as array(varchar))) AS t(app_list)
            GROUP BY 1,2,3
    """
    
    df_temp = pd.read_sql(users_app_list, connection)
    df_temp['app_list'] = df_temp['app_list'].str.strip()
    ## Less memory usage 
    df_temp = df_temp.groupby(['last_sync_date', 'user_id']).agg(app_list_set = ('app_list' , list)).reset_index() 
    df_temp['last_sync_date'] = df_temp['last_sync_date'].astype('int32') 
    df_temp.to_csv(local_datasets + 'users_app_list_{}.csv'.format(date_str), index=False)
    end_time = time.time()
    execution_time = ((end_time - start_time)/60.00)
    print(date_str, " - execution time : ", execution_time, " mins")
    

#Reset the index of the final DataFrame
#all_data.reset_index(drop=True, inplace=True)

20240630  - execution time :  8.187083796660106  mins
20240701  - execution time :  15.338780117034911  mins


KeyboardInterrupt: 

## Reading Local Extract

In [5]:
csv_files = [file for file in os.listdir(local_datasets) if file.endswith('.csv')]
sorted(csv_files)

['users_app_list_20240218.csv',
 'users_app_list_20240219.csv',
 'users_app_list_20240220.csv',
 'users_app_list_20240221.csv',
 'users_app_list_20240222.csv',
 'users_app_list_20240223.csv',
 'users_app_list_20240224.csv',
 'users_app_list_20240225.csv',
 'users_app_list_20240226.csv',
 'users_app_list_20240227.csv',
 'users_app_list_20240228.csv',
 'users_app_list_20240229.csv',
 'users_app_list_20240301.csv',
 'users_app_list_20240302.csv',
 'users_app_list_20240303.csv',
 'users_app_list_20240304.csv',
 'users_app_list_20240305.csv',
 'users_app_list_20240306.csv',
 'users_app_list_20240307.csv',
 'users_app_list_20240308.csv',
 'users_app_list_20240309.csv',
 'users_app_list_20240310.csv',
 'users_app_list_20240311.csv',
 'users_app_list_20240312.csv',
 'users_app_list_20240313.csv',
 'users_app_list_20240314.csv',
 'users_app_list_20240315.csv',
 'users_app_list_20240316.csv',
 'users_app_list_20240317.csv',
 'users_app_list_20240318.csv',
 'users_app_list_20240319.csv',
 'users_

In [6]:
csv_files = ['users_app_list_20240501.csv','users_app_list_20240502.csv','users_app_list_20240503.csv', 'users_app_list_20240504.csv',
 'users_app_list_20240505.csv',
 'users_app_list_20240506.csv', 'users_app_list_20240507.csv','users_app_list_20240508.csv', 'users_app_list_20240509.csv',
 'users_app_list_20240510.csv', 'users_app_list_20240511.csv', 'users_app_list_20240512.csv', 'users_app_list_20240513.csv',
 'users_app_list_20240514.csv', 'users_app_list_20240515.csv',
 'users_app_list_20240516.csv', 'users_app_list_20240517.csv', 'users_app_list_20240518.csv', 'users_app_list_20240519.csv',
 'users_app_list_20240520.csv', 'users_app_list_20240521.csv', 'users_app_list_20240522.csv', 'users_app_list_20240523.csv',
 'users_app_list_20240524.csv', 'users_app_list_20240525.csv', 'users_app_list_20240526.csv', 'users_app_list_20240527.csv',
 'users_app_list_20240528.csv', 'users_app_list_20240529.csv', 'users_app_list_20240530.csv', 'users_app_list_20240531.csv',
 'users_app_list_20240601.csv', 'users_app_list_20240602.csv', 'users_app_list_20240603.csv',
 'users_app_list_20240604.csv', 'users_app_list_20240605.csv', 'users_app_list_20240606.csv', 'users_app_list_20240607.csv',
 'users_app_list_20240608.csv', 'users_app_list_20240609.csv', 'users_app_list_20240610.csv', 'users_app_list_20240611.csv',
 'users_app_list_20240612.csv', 'users_app_list_20240613.csv', 'users_app_list_20240614.csv', 'users_app_list_20240615.csv',
 'users_app_list_20240616.csv', 'users_app_list_20240617.csv', 'users_app_list_20240618.csv',
 'users_app_list_20240619.csv', 'users_app_list_20240620.csv', 'users_app_list_20240621.csv', 'users_app_list_20240622.csv',
 'users_app_list_20240623.csv', 'users_app_list_20240624.csv', 'users_app_list_20240625.csv', 'users_app_list_20240626.csv',
 'users_app_list_20240627.csv', 'users_app_list_20240628.csv', 'users_app_list_20240629.csv', 'users_app_list_20240630.csv',
 'users_app_list_20240701.csv']

In [7]:
# csv_files = [file for file in os.listdir(local_datasets) if file.endswith('.csv')]

dfs = []

# Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    start_time = time.time()
    file_path = os.path.join(local_datasets, file)
    df = pd.read_csv(file_path)
    dfs.append(df)
    end_time = time.time()
    execution_time = ((end_time - start_time)/60.00)
    print(file_path, " - execution time : ", execution_time, " mins")

# Concatenate all DataFrames in the list into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.head(2)

/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240501.csv  - execution time :  0.05894264777501424  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240502.csv  - execution time :  0.05371652046839396  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240503.csv  - execution time :  0.056584084033966066  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240504.csv  - execution time :  0.06026393175125122  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240505.csv  - execution time :  0.06182954708735148  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240506.csv  - execution time :  0.06937758127848308  mins
/Users/E2074/local-datasets/customer/appography/appography_data_dump/users_app_list_20240507.csv  - execution time :  0.061223200956980

,last_sync_date,user_id,app_list_set
0,20240430,5737c70bddbec2203f7334a0,"['messenger', 'whatsapp', 'wynk music', 'gmail..."
1,20240430,573f28f39b0ffc283676f0b7,"['gmail', 'whatsapp', 'bookmyshow', 'google ca..."


In [8]:
combined_df.shape

(63590645, 3)

In [9]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63590645 entries, 0 to 63590644
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int64 
 1   user_id         object
 2   app_list_set    object
dtypes: int64(1), object(2)
memory usage: 1.4+ GB


In [10]:
combined_df['last_sync_date'] = combined_df['last_sync_date'].astype('int32')

In [11]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63590645 entries, 0 to 63590644
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 1.2+ GB


In [12]:
combined_df.shape

(63590645, 3)

In [13]:
combined_df_sorted = combined_df.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])

In [14]:
combined_df_sorted.head(2)

,last_sync_date,user_id,app_list_set
55106733,20240623,5737c6adddbec2203f73316d,"['paytm money', 'instagram', 'disney+ hotstar'..."
53114835,20240622,5737c6adddbec2203f73316d,"['youtube', 'wynk music', 'borzo driver app', ..."


In [15]:
combined_df_latest = combined_df_sorted.drop_duplicates(subset='user_id', keep='first')

In [16]:
combined_df_latest.head(2)

,last_sync_date,user_id,app_list_set
55106733,20240623,5737c6adddbec2203f73316d,"['paytm money', 'instagram', 'disney+ hotstar'..."
61403117,20240630,5737c6aeddbec2203f733176,"['airtel thanks', 'zomato', 'cred', 'facebook'..."


In [17]:
combined_df_sorted.shape

(63590645, 3)

In [18]:
combined_df_latest.shape

(27933634, 3)

In [19]:
combined_df_sorted.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63590645 entries, 55106733 to 62496188
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 1.7+ GB


In [20]:
combined_df_latest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27933634 entries, 55106733 to 62496188
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 745.9+ MB


In [21]:
combined_df_latest.user_id.nunique()

27933634

## Clear Other Data

In [22]:
# dfs = None
# combined_df = None
# combined_df_sorted = None

## Refined Data

In [24]:
combined_df_latest.to_csv('/Users/E2074/local-datasets/customer/appography/appography_data_last_sync/appography-data-may-june.csv', index=False)

In [21]:
# combined_df_latest.to_csv('/Users/rapido/local-datasets/customer/appography/appography_data_last_sync/appography-data-june08.csv', index=False)

In [27]:
# combined_df_latest.to_csv('/Users/rapido/local-datasets/customer/appography/appography_data_last_sync/appography-data-march-april.csv', index=False)

In [28]:
# combined_df_latest = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_data_last_sync/appography-data-march-april.csv')

In [29]:
combined_df_latest

,last_sync_date,user_id,app_list_set
0,20240316,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
1,20240422,5737c6adddbec2203f73316d,"['messenger', 'wynk music', 'google photos', '..."
2,20240423,5737c6aeddbec2203f733176,"['whatsapp', 'google maps', 'gmail', 'zomato',..."
3,20240411,5737c6b0ddbec2203f733182,"['jio cinema', 'disney+ hotstar', 'paytm money..."
4,20240321,5737c6b1ddbec2203f73318b,"['snapchat', 'instagram', 'google photos', 'zo..."
...,...,...,...
22676863,20240426,662bf27848dd5273a26dc865,"['jio cinema', 'twitter', 'disney+ hotstar', '..."
22676864,20240426,662bf27a9b5b3e0c5569b286,"['driveu driver app', 'swiggy', 'instagram', '..."
22676865,20240426,662bf27d48dd52cb386dc948,"['spotify', 'whatsapp', 'snapchat', 'clock', '..."
22676866,20240426,662bf2809b5b3e46d969b3a2,"['google sheets', 'google calendar', 'instagra..."


In [30]:
combined_df_latest.user_id.nunique()

22676868

## Further Analysis

In [31]:
df_expand = combined_df_latest.head(5000)
df_expand.head(5)

,last_sync_date,user_id,app_list_set
0,20240316,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
1,20240422,5737c6adddbec2203f73316d,"['messenger', 'wynk music', 'google photos', '..."
2,20240423,5737c6aeddbec2203f733176,"['whatsapp', 'google maps', 'gmail', 'zomato',..."
3,20240411,5737c6b0ddbec2203f733182,"['jio cinema', 'disney+ hotstar', 'paytm money..."
4,20240321,5737c6b1ddbec2203f73318b,"['snapchat', 'instagram', 'google photos', 'zo..."


In [32]:
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_expand['app_list_set'] = df_expand['app_list_set'].apply(lambda x: extract_and_trim(x))

In [33]:
df_expand

,last_sync_date,user_id,app_list_set
0,20240316,5737c6acddbec2203f73315e,"[flipkart, disney+ hotstar, messenger, meesho,..."
1,20240422,5737c6adddbec2203f73316d,"[messenger, wynk music, google photos, google ..."
2,20240423,5737c6aeddbec2203f733176,"[whatsapp, google maps, gmail, zomato, cred, m..."
3,20240411,5737c6b0ddbec2203f733182,"[jio cinema, disney+ hotstar, paytm money, ins..."
4,20240321,5737c6b1ddbec2203f73318b,"[snapchat, instagram, google photos, zoom, my1..."
...,...,...,...
4995,20240409,573f29159b0ffc283677734a,"[whatsapp, wynk music, rapido captain, ola dri..."
4996,20240421,573f29159b0ffc283677734b,"[microsoft 365, outlook, aha, airbnb, redbus, ..."
4997,20240303,573f29159b0ffc2836777351,"[linkedin, bookmyshow, google maps, twitter, u..."
4998,20240426,573f29159b0ffc2836777353,"[zee5, whatsapp, flipkart, onedrive, youtube, ..."


In [34]:
unique_categories = set(category for sublist in df_expand['app_list_set'] for category in sublist)

for category in unique_categories:
    df_expand[category] = df_expand['app_list_set'].apply(lambda x: 'yes' if category in x else 'no')

In [35]:
df_expand.head()

,last_sync_date,user_id,app_list_set,viber,google calendar,google slides,simplilearn,google analytics,sharechat,blusmart,truecaller,huawei health,calculator,webex,instagram,zoho mail,spotify,jiomart,strava,meesho,garmin connect,amazon shopping,confluence,tumblr,microsoft 365,khan academy,photomath,outlook,hike,edx,goibibo,onedrive,geeks for geeks,twitter,voot,rapido captain,whatsapp,microsoft math solver,ms edge,economic times,ola,google photos,disney+ hotstar,driveu driver app,hangouts,chegg study,freecharge,file commander,cleartrip,mlb,microsoft excel,dunzo,zoho meeting,mobikwik,miro,aisle,brainly,mx takatak,doubtnut,linkedin,bigbasket,groww,google authenticator,duolingo,pocket,uber eats,ludo king,hinge,mint,roposo,wynk music,ncert books,coffee meets bagel,shiprocket driver app,alt balaji,bookmyshow,wechat,fiton,firstcry,jira,udemy,pharmeasy,happn,hdfc mobile banking,coursera,zivame,snapchat,shiksha.com,paytm mall,zepto driver app,amazon music,google maps,byju,zomato driver app,pinterest,indigo,kite by zerodha,accuweather,microsoft onenote,youtube,telegram,myntra,borzo driver app,quickride,line,trulymadly,ajio,zee5,nike training club,tinder,bumble,zomato,uber,ignou e-content,axis mobile,google sheets,nykaa,picsart,whitehat jr,my11circle,grofers,zoom,paytm money,adobe lightroom,asana,jio cinema,unacademy,google classroom,gaana,fitbit,swiggy driver app,trello,feedly,kayak,amazon prime video,physics wallah,zepp (amazfit),adobe scan,mycbseguide,porter driver app,namma yatri,grofers delivery app,dunzo driver app,moodle,slack,booking.com,nba,ola driver app,in drive,netflix,namma partner,fossil smartwatches,caclubindia,cisco,microsoft lens,aha,skype,flipkart,aakash,flipboard,myteam11,airbnb,facebook,sonyliv,swiggy,ms authenticator,swayam,tripadvisor,mi fit,trivago,diksha,google news,messenger,uber driver app,quick ride,goodreads,okcupid,intune company portal,healthifyme,jugnoo,dropbox,quackquack,qr code reader,shadowfax driver app,clock,redbus,notion,myfitnesspal,vision11,gmail,soundcloud,cred,samsung galaxy wearable,the indian express,vedantu,angel broking,nps,e-pg pathshala,github,club factory,yatra,hungama music,microsoft teams
0,20240316,5737c6acddbec2203f73315e,"[flipkart, disney+ hotstar, messenger, meesho,...",no,yes,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,yes,yes,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,yes,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,no,no,no,no,yes,no,yes,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no
1,20240422,5737c6adddbec2203f73316d,"[messenger, wynk music, google photos, google ...",no,no,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,no,no,yes,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,yes,no,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,yes,no,yes,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,no,no,no,no,no,no,no
2,20240423,5737c6aeddbec2203f733176,"[whatsapp, google maps, gmail, zomato, cred, m...",no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,yes,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,

In [36]:
df_expand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 203 entries, last_sync_date to microsoft teams
dtypes: int64(1), object(202)
memory usage: 7.7+ MB


In [37]:
combined_df_latest.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22676868 entries, 0 to 22676867
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int64 
 1   user_id         object
 2   app_list_set    object
dtypes: int64(1), object(2)
memory usage: 519.0+ MB


## Temp Datasets

In [38]:
df_expand.head()

,last_sync_date,user_id,app_list_set,viber,google calendar,google slides,simplilearn,google analytics,sharechat,blusmart,truecaller,huawei health,calculator,webex,instagram,zoho mail,spotify,jiomart,strava,meesho,garmin connect,amazon shopping,confluence,tumblr,microsoft 365,khan academy,photomath,outlook,hike,edx,goibibo,onedrive,geeks for geeks,twitter,voot,rapido captain,whatsapp,microsoft math solver,ms edge,economic times,ola,google photos,disney+ hotstar,driveu driver app,hangouts,chegg study,freecharge,file commander,cleartrip,mlb,microsoft excel,dunzo,zoho meeting,mobikwik,miro,aisle,brainly,mx takatak,doubtnut,linkedin,bigbasket,groww,google authenticator,duolingo,pocket,uber eats,ludo king,hinge,mint,roposo,wynk music,ncert books,coffee meets bagel,shiprocket driver app,alt balaji,bookmyshow,wechat,fiton,firstcry,jira,udemy,pharmeasy,happn,hdfc mobile banking,coursera,zivame,snapchat,shiksha.com,paytm mall,zepto driver app,amazon music,google maps,byju,zomato driver app,pinterest,indigo,kite by zerodha,accuweather,microsoft onenote,youtube,telegram,myntra,borzo driver app,quickride,line,trulymadly,ajio,zee5,nike training club,tinder,bumble,zomato,uber,ignou e-content,axis mobile,google sheets,nykaa,picsart,whitehat jr,my11circle,grofers,zoom,paytm money,adobe lightroom,asana,jio cinema,unacademy,google classroom,gaana,fitbit,swiggy driver app,trello,feedly,kayak,amazon prime video,physics wallah,zepp (amazfit),adobe scan,mycbseguide,porter driver app,namma yatri,grofers delivery app,dunzo driver app,moodle,slack,booking.com,nba,ola driver app,in drive,netflix,namma partner,fossil smartwatches,caclubindia,cisco,microsoft lens,aha,skype,flipkart,aakash,flipboard,myteam11,airbnb,facebook,sonyliv,swiggy,ms authenticator,swayam,tripadvisor,mi fit,trivago,diksha,google news,messenger,uber driver app,quick ride,goodreads,okcupid,intune company portal,healthifyme,jugnoo,dropbox,quackquack,qr code reader,shadowfax driver app,clock,redbus,notion,myfitnesspal,vision11,gmail,soundcloud,cred,samsung galaxy wearable,the indian express,vedantu,angel broking,nps,e-pg pathshala,github,club factory,yatra,hungama music,microsoft teams
0,20240316,5737c6acddbec2203f73315e,"[flipkart, disney+ hotstar, messenger, meesho,...",no,yes,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,yes,yes,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,yes,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,no,no,no,no,yes,no,yes,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no
1,20240422,5737c6adddbec2203f73316d,"[messenger, wynk music, google photos, google ...",no,no,no,no,no,no,no,yes,no,no,no,yes,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,no,no,yes,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,yes,no,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,yes,no,yes,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,no,no,no,no,no,no,no
2,20240423,5737c6aeddbec2203f733176,"[whatsapp, google maps, gmail, zomato, cred, m...",no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,no,no,no,no,yes,no,yes,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,no,yes,no,no,no,no,no,no,

### Explode and Map category 

In [43]:
df_expand_app_list = df_expand[['user_id', 'app_list_set']]
# df_expand_app_list.reset_index(drop=True, inplace=True)

df_expand_app_list = df_expand_app_list.explode('app_list_set')
df_expand_app_list['app_list_set'] = df_expand_app_list['app_list_set'].str.lower()

df_expand_app_list.head(1)

,user_id,app_list_set
0,5737c6acddbec2203f73315e,flipkart


In [44]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['Educational' 'OTT' 'Delivery_Grocery' 'Tools' 'Office' 'Dating' 'News'
 'Devotional' 'Travel_Metro' 'Gaming' 'Travel_Bookings'
 'Finance_Transactions' 'Ecommerce' 'Streaming_Music' 'Finance_Investment'
 'Health' 'Fantasy_Sports' 'Delivery_Food' 'Home_Security' 'Finance_News'
 'Travel_Ridehailing' 'Social' 'Telecom' 'Entertainment' 'News_finance'
 'Wearable' 'Driver_App']
['Educational' 'Rest' 'Office' 'Ride haling' 'Gaming' 'Finance_Investment'
 'Driver_App']


,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational


In [61]:
## merge

df_expand_app_list_category =  pd.merge(df_expand_app_list,
                                        category_list, 
                                        how='left', 
                                        left_on='app_list_set',  
                                        right_on='app_name'
                                       )
df_expand_app_list_category

,user_id,app_list_set,app_name,categories_l2,categories_l1
0,5737c6acddbec2203f73315e,flipkart,flipkart,Rest,Ecommerce
1,5737c6acddbec2203f73315e,disney+ hotstar,disney+ hotstar,Rest,OTT
2,5737c6acddbec2203f73315e,messenger,messenger,Rest,Social
3,5737c6acddbec2203f73315e,meesho,meesho,Rest,Ecommerce
4,5737c6acddbec2203f73315e,truecaller,truecaller,Rest,Tools
...,...,...,...,...,...
182932,573f29209b0ffc283677942b,paytm money,paytm money,Rest,Finance_Transactions
182933,573f29209b0ffc283677942b,adobe scan,adobe scan,Rest,Tools
182934,573f29209b0ffc283677942b,whatsapp,whatsapp,Rest,Social
182935,573f29209b0ffc283677942b,ola,ola,Ride haling,Travel_Ridehailing


In [62]:
df_customer_mapped = df_expand_app_list_category\
                        .groupby(['user_id'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
#                              app_count = ('app_list_set', 'nunique'),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
#                              categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', lambda x: list(set(x)))
#                              categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

print(df_customer_mapped.shape)
df_customer_mapped.head(5)

(5000, 4)


,user_id,app_list,categories_l1,categories_l2
0,5737c6acddbec2203f73315e,"[paytm money, google news, swiggy, gmail, mess...","[Ecommerce, Tools, OTT, Travel_Ridehailing, So...","[Ride haling, Rest]"
1,5737c6adddbec2203f73316d,"[snapchat, paytm money, gmail, google sheets, ...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Office, Driver_App, Rest]"
2,5737c6aeddbec2203f733176,"[paytm money, ms edge, gmail, messenger, bigba...","[Finance_Investment, Tools, OTT, Travel_Rideha...","[Ride haling, Finance_Investment, Office, Rest]"
3,5737c6b0ddbec2203f733182,"[voot, paytm money, swiggy, gmail, messenger, ...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Driver_App, Rest]"
4,5737c6b1ddbec2203f73318b,"[snapchat, clock, paytm money, twitter, google...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Office, Driver_App, Rest]"


In [63]:
total_customers = df_customer_mapped.user_id.nunique()
total_customers

5000

#### Distribution check on categories

In [64]:
df_temp = df_expand_app_list_category\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 app_count = ('app_list_set', 'nunique'),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,app_count,customers,%
0,Rest,"[snapchat, voot, clock, myteam11, paytm money,...",123,5000,100.00
1,Ride haling,"[uber, quickride, namma yatri, blusmart, ola, ...",9,4516,90.32
2,Office,"[pocket, webex, dropbox, google authenticator,...",22,4082,81.64
3,Finance_Investment,"[namma partner, kite by zerodha, hdfc mobile b...",6,3437,68.74
4,Educational,"[goodreads, pocket, diksha, google classroom, ...",30,1143,22.86
5,Gaming,[ludo king],1,582,11.64
6,Driver_App,"[zepto driver app, rapido captain, zomato driv...",10,504,10.08


In [65]:
df_temp = df_expand_app_list_category\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 app_count = ('app_list_set', 'nunique'),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Tools,"[notion, calculator, clock, tumblr, ms edge, g...",23,5000,100.00
1,Social,"[snapchat, roposo, wechat, viber, twitter, hik...",18,4993,99.86
2,Travel_Ridehailing,"[uber, quickride, namma yatri, blusmart, ola, ...",9,4516,90.32
3,OTT,"[voot, jio cinema, disney+ hotstar, amazon pri...",9,4513,90.26
4,Finance_Transactions,"[paytm money, mobikwik, freecharge, axis mobil...",5,4504,90.08
5,Delivery_Food,"[dunzo, swiggy, uber eats, zomato]",4,4298,85.96
6,Ecommerce,"[zivame, myntra, jiomart, club factory, firstc...",13,4264,85.28
7,Office,"[pocket, webex, dropbox, google authenticator,...",22,4082,81.64
8,Streaming_Music,"[spotify, wynk music, hungama music, soundclou...",6,3795,75.90
9,Finance_Investment,"[namma partner, kite by zerodha, hdfc mobile b...",6,3437,68.74


#### Category-level App usage 

In [66]:
### Office
df_temp = df_expand_app_list_category[~df_expand_app_list_category['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('user_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,3919,78.38
1,Ride haling,uber,3618,72.36
2,Ride haling,namma yatri,1534,30.68
3,Ride haling,quick ride,590,11.80
4,Ride haling,blusmart,565,11.30
5,Ride haling,in drive,395,7.90
6,Ride haling,driveu driver app,35,0.70
7,Ride haling,jugnoo,16,0.32
8,Ride haling,quickride,11,0.22
9,Office,zoom,2041,40.82


#### Transform Categories level-2

In [67]:
df_customer_mapped.head(1)

,user_id,app_list,categories_l1,categories_l2
0,5737c6acddbec2203f73315e,"[paytm money, google news, swiggy, gmail, mess...","[Ecommerce, Tools, OTT, Travel_Ridehailing, So...","[Ride haling, Rest]"


In [68]:
unique_categories = set(category for sublist in df_customer_mapped['categories_l1'] for category in sublist)
# print(unique_categories)

for category in unique_categories:
    df_customer_mapped[category] = df_customer_mapped['categories_l1']\
                                            .apply(lambda x: 1 if category in x else 0)
##When havnig nan last column
# df_customer_mapped.drop(df_customer_mapped.columns[-1], axis=1, inplace=True)
df_customer_mapped.head(1)

,user_id,app_list,categories_l1,categories_l2,OTT,Travel_Ridehailing,Dating,Finance_Transactions,News,Ecommerce,Finance_Investment,Gaming,Social,Driver_App,Office,Tools,Devotional,Educational,Travel_Bookings,Streaming_Music,Wearable,Fantasy_Sports,Health,Entertainment,Delivery_Grocery,Delivery_Food,Finance_News
0,5737c6acddbec2203f73315e,"[paytm money, google news, swiggy, gmail, mess...","[Ecommerce, Tools, OTT, Travel_Ridehailing, So...","[Ride haling, Rest]",1,1,0,1,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0


In [69]:
df_customer_mapped

,user_id,app_list,categories_l1,categories_l2,OTT,Travel_Ridehailing,Dating,Finance_Transactions,News,Ecommerce,Finance_Investment,Gaming,Social,Driver_App,Office,Tools,Devotional,Educational,Travel_Bookings,Streaming_Music,Wearable,Fantasy_Sports,Health,Entertainment,Delivery_Grocery,Delivery_Food,Finance_News
0,5737c6acddbec2203f73315e,"[paytm money, google news, swiggy, gmail, mess...","[Ecommerce, Tools, OTT, Travel_Ridehailing, So...","[Ride haling, Rest]",1,1,0,1,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0
1,5737c6adddbec2203f73316d,"[snapchat, paytm money, gmail, google sheets, ...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Office, Driver_App, Rest]",1,1,0,1,0,1,0,0,1,1,1,1,0,0,0,1,1,0,0,0,0,0,0
2,5737c6aeddbec2203f733176,"[paytm money, ms edge, gmail, messenger, bigba...","[Finance_Investment, Tools, OTT, Travel_Rideha...","[Ride haling, Finance_Investment, Office, Rest]",1,1,0,1,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0,1,1,0
3,5737c6b0ddbec2203f733182,"[voot, paytm money, swiggy, gmail, messenger, ...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Driver_App, Rest]",1,1,0,1,0,1,0,0,1,1,0,1,0,0,0,1,0,0,0,1,0,1,0
4,5737c6b1ddbec2203f73318b,"[snapchat, clock, paytm money, twitter, google...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Office, Driver_App, Rest]",1,1,0,1,1,1,0,0,1,1,1,1,0,0,1,0,0,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,573f29209b0ffc283677941c,"[webex, twitter, airbnb, ms edge, paytm money,...","[Tools, Ecommerce, OTT, Travel_Ridehailing, So...","[Ride haling, Office, Rest]",1,1,0,1,0,1,0,0,1,0,1,1,0,0,1,1,1,0,0,1,0,1,0
4996,573f29209b0ffc283677941d,"[paytm money, swiggy, google news, gmail, shar...","[Ecommerce, Tools, OTT, Travel_Ridehailing, Ga...","[Ride haling, Gaming, Rest]",1,1,0,1,1,1,0,1,1,0,0,1,0,0,0,1,0,0,0,0,1,1,0
4997,573f29209b0ffc2836779421,"[strava, hdfc mobile banking, swiggy, google n...","[Tools, Finance_Investment, OTT, Travel_Rideha...","[Finance_Investment, Ride haling, Educational,...",1,1,0,0,1,0,1,0,1,0,1,1,0,1,0,1,0,0,1,0,0,1,0
4998,573f29209b0ffc2836779427,"[clock, swiggy, gmail, sharechat, google sheet...","[Tools, OTT, Travel_Ridehailing, Dating, Socia...","[Ride haling, Office, Rest]",1,1,1,0,0,0,0,0,1,0,1,1,0,0,1,1,0,0,0,1,1,1,0
